# Pancreas and pancreatic lymph node: tissue-specific protein profiles

Compare sample coverage, cell identities and protein measurements across pancreas and pancreatic lymph node (pLN). Each tissue has its own native typing system, normalization and UMAP. The pancreas carries final hierarchy assignments; pLN identities are candidates from its separate protein rules.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/05_tissue_comparisons.ipynb).

## 1. Load the tissue-specific samples

The dataset includes all available pancreas donors and only the subset with pLN tissue. Select a donor with both tissues and a marker they share.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "pancreas_islets.h5ad").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the processed PhenoCycler data directory.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})
DONOR = "115"
MARKER = "CD3e"
pan = sc.read_h5ad(data / "pancreas_umap.h5ad")
pln = sc.read_h5ad(data / "pln_umap.h5ad")
counts = pd.read_parquet(data / "donor_counts.parquet")
counts["donor"] = counts.donor.astype(str)
n = counts.loc[counts.level.eq("primary")].groupby(["tissue","donor"],observed=True).eligible_cells.first().unstack("tissue")
paired = n.dropna().index
if DONOR not in paired: raise ValueError(f"Choose a donor with both tissues: {sorted(paired,key=int)}")
if MARKER not in set(pan.var_names) & set(pln.var_names): raise ValueError("Choose a shared measured marker.")
print(f"{n.Pancreas.notna().sum()} donors with pancreas · {n.pLN.notna().sum()} with pLN · {len(paired)} paired donors")
display(n.loc[[DONOR]].astype(int).rename(columns={"Pancreas":"Admitted pancreas cells","pLN":"Admitted pLN cells"}))

## 2. Visualize the independent tissue embeddings

Each panel shows the selected donor in the shared coordinate system for that tissue. Distances between panels have no biological interpretation. Small populations are enriched in the display sample; do not count dots to estimate abundance.

In [ ]:
from matplotlib.lines import Line2D
def view(a, donor):
    keep = a.obs.donor.astype(str).eq(donor).to_numpy() & np.isfinite(a.obsm["X_umap"]).all(axis=1)
    return a[keep].copy()
sections = [("Pancreas",view(pan,DONOR)), ("pLN",view(pln,DONOR))]
fig, axs = plt.subplots(1,2,figsize=(12.3,5.7))
colors = ["#294e77","#bf6a2c","#348472","#9062a5","#ac4e62","#86734c","#608aa5","#737373","#ba983c","#394c53","#a791b4"]
for (t,a),ax in zip(sections,axs):
    labels = a.obs.primary.astype("category").cat.remove_unused_categories()
    a.obs["Tissue assignment"] = labels
    palette = dict(zip(labels.cat.categories,colors))
    sc.pl.umap(a,color="Tissue assignment",ax=ax,palette=palette,size=12,frameon=False,
               legend_loc="none",show=False,title=f"{t}  ·  donor {DONOR}  ·  {a.n_obs:,} mapped cells")
    handles=[Line2D([],[],marker="o",linestyle="",color=palette[x],label=x,markersize=5) for x in labels.cat.categories]
    ax.legend(handles=handles,loc="upper center",bbox_to_anchor=(.5,-.02),ncol=2,frameon=False,fontsize=8)
fig.subplots_adjust(bottom=.30,top=.9,wspace=.14)
plt.show()

## 3. Compare one protein in its tissue context

Pancreas uses the cohort aligned floor/span typing layer. pLN uses its own section-specific clipped scale. Each color bar refers to values in its own tissue; do not interpret the color difference as a fold-change. Missing measurements are shown in grey.

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(10.5,4.2))
for (t,a),ax in zip(sections,axs):
    layer = "typing_scaled" if t=="Pancreas" else None
    m = np.asarray((a.layers[layer] if layer else a.X)[:,a.var_names.get_loc(MARKER)]).ravel()
    if np.isfinite(m).any():
        sc.pl.umap(a,color=MARKER,layer=layer,use_raw=False,ax=ax,cmap="viridis",na_color="#c6cace",
                   size=14,vmin=0,vmax=1,frameon=False,show=False,
                   title=f"{t}  ·  {MARKER}  ·  {np.isfinite(m).sum():,} measured")
    else:
        ax.text(.5,.5,"Marker unavailable",ha="center",va="center",transform=ax.transAxes);ax.set_axis_off()
fig.tight_layout()
plt.show()

## 4. Compare cell composition within each tissue

Bars use every admitted cell of the selected donor in that tissue. No-rule-match, ambiguous and unavailable pLN candidates remain in the denominator; their counts are displayed alongside named pLN categories. Shared words such as “Immune” do not imply matched gates or identical parent populations.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12.1,5.4),sharex=True)
for ax,t,c in zip(axes,["Pancreas","pLN"],["#397888","#94704c"]):
    z=counts.loc[counts.tissue.eq(t)&counts.donor.eq(DONOR)&counts.level.eq("primary")].copy()
    denom = int(z.eligible_cells.iloc[0])
    assert int(z.cells.sum()) == denom
    z=z[z.cells.gt(0)].sort_values("cells")
    share=100*z.cells/denom
    ax.barh(z.label,share,color=c,height=.66)
    ax.set(xlim=(0,105),xlabel="Admitted cells in this tissue (%)",
           title=f"{t}  ·  {denom:,} cells")
    ax.bar_label(ax.containers[0],labels=[f"{v:.2g}%" for v in share],padding=3,size=8)
fig.tight_layout(w_pad=2)
plt.show()